# AMJax — PyAMG Solver Compatibility Benchmark

Benchmarks every PyAMG hierarchy factory as input to `AMJAXSolver.from_pyamg`.
For each solver, the PyAMG hierarchy is built, converted to JAX, then timed with
V-cycle and PCG (AMG-preconditioned CG) solves on batches of 2D Poisson RHS.

Results and plots are saved to `results/`.


[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vboussange/AMJax/blob/main/benchmarks/rss.ipynb)

In [ ]:
import timeit

import numpy as np
import jax
import jax.numpy as jnp
import pyamg

from amjax import AMJAXSolver
from benchmarks.params import (
    TOL as tol,
    MAXITER_VCYCLE as maxiter_vcycle,
    MAXITER_SOLV as maxiter_solv,
    GRID_SIZE as grid_size,
    VMAP_K,
    SOLVERS_COMPARAISON,
)
from benchmarks.plots import (
    plot_runtime,
    plot_residual,
    save_results,
    load_results,
)

jax.config.update("jax_enable_x64", True)
np.random.seed(42)

print(f"JAX backend : {jax.devices()[0].platform}")
print(f"Grid sizes  : {grid_size}")
print(f"Batch size  : {VMAP_K}")


## Solvers

| Name | Module | Intended for | Docs |
|------|--------|--------------|------|
| `smoothed_aggregation` | `pyamg.aggregation` | SPD with known near-null space | [docs](https://pyamg.readthedocs.io/en/latest/generated/pyamg.aggregation.smoothed_aggregation_solver.html) |
| `rootnode` | `pyamg.aggregation` | Near-singular / non-diag-dominant | [docs](https://pyamg.readthedocs.io/en/latest/generated/pyamg.aggregation.rootnode_solver.html) |
| `pairwise` | `pyamg.aggregation` | Low setup cost, greedy aggregation | [docs](https://pyamg.readthedocs.io/en/latest/generated/pyamg.aggregation.pairwise_solver.html) |
| `ruge_stuben` | `pyamg.classical` | General sparse systems | [docs](https://pyamg.readthedocs.io/en/latest/generated/pyamg.classical.ruge_stuben_solver.html) |
| `air` | `pyamg.classical` | Non-symmetric systems (AIR) | [docs](https://pyamg.readthedocs.io/en/latest/generated/pyamg.classical.air_solver.html) |

> **AMJax limitations shared by all solvers:** V-cycle only · `jacobi` coarse solver only.


In [ ]:
SOLVERS = {
    "smoothed_aggregation": pyamg.smoothed_aggregation_solver,
    "rootnode": pyamg.rootnode_solver,
    "pairwise": pyamg.pairwise_solver,
    "ruge_stuben": pyamg.ruge_stuben_solver,
    "air": pyamg.air_solver,
}

SOLVERS = {k: v for k, v in SOLVERS.items() if k in SOLVERS_COMPARAISON}

## Benchmark

V-cycle and PCG solves, batched over K right-hand sides, JIT-compiled and vmap-ed.


In [4]:
@jax.jit
def amjax_solve(ml, b):
    x = ml.solve(b, tol=tol, maxiter=maxiter_vcycle)
    error = jnp.linalg.norm(b - ml.levels[0].A @ x) / jnp.linalg.norm(b)
    return x, error


@jax.jit
def amjax_pcg_solve(ml, b):
    M = ml.aspreconditioner()
    x, _ = jax.scipy.sparse.linalg.cg(ml.levels[0].A, b, M=M, tol=tol, maxiter=maxiter_solv)
    error = jnp.linalg.norm(b - ml.levels[0].A @ x) / jnp.linalg.norm(b)
    return x, error


def amjax_vmap_solve_batch(ml, B):
    results = jax.vmap(lambda b: amjax_solve(ml, b))(B)
    return results[0], float(jnp.mean(results[1]))


def amjax_pcg_vmap_solve_batch(ml, B):
    results = jax.vmap(lambda b: amjax_pcg_solve(ml, b))(B)
    return results[0], float(jnp.mean(results[1]))


def benchmark(method, func, is_jax=True):
    if is_jax:
        jax.block_until_ready(func())
    times = timeit.repeat(func, number=1, repeat=10)
    _, error = func()
    print(f"{method}/ time: {min(times):.4f}s/ residual: {float(error):.2e}")
    return min(times), float(error)


In [6]:
time_vcycle = {name: [] for name in SOLVERS}
time_pcg = {name: [] for name in SOLVERS}
res_vcycle = {name: [] for name in SOLVERS}
res_pcg = {name: [] for name in SOLVERS}

for n in grid_size:
    print(f"\nGrid size: {n}x{n}")
    A  = pyamg.gallery.poisson((n, n), format="csr")
    B  = np.random.rand(VMAP_K, A.shape[0])
    B_jx = jnp.array(B)

    for name, factory in SOLVERS.items():
        try:
            ml_jax = AMJAXSolver.from_pyamg(factory(A, coarse_solver="jacobi"))

            t, r = benchmark(
                f"AMJAX-{name} vmap",
                lambda ml=ml_jax: amjax_vmap_solve_batch(ml, B_jx),
            )
            time_vcycle[name].append(t)
            res_vcycle[name].append(r)

            t, r = benchmark(
                f"AMJAX-{name}-PCG vmap",
                lambda ml=ml_jax: amjax_pcg_vmap_solve_batch(ml, B_jx),
            )
            time_pcg[name].append(t)
            res_pcg[name].append(r)

        except Exception as exc:
            for d in (time_vcycle, time_pcg, res_vcycle, res_pcg):
                d[name].append(float("nan"))
            print(f"AMJAX-{name}: ERROR — {exc}")


In [ ]:
save_results({
    "solver_names": list(SOLVERS.keys()),
    "grid_size":    grid_size,
    "vmap_k":       VMAP_K,
    "time_vcycle":  time_vcycle,
    "time_pcg":     time_pcg,
    "res_vcycle":   res_vcycle,
    "res_pcg":      res_pcg,
}, "solvers_benchmark.json")


## Results

In [7]:
W = max(len(name) for name in SOLVERS) + 2
print(f"\n{'=' * 76}")
print(f"  Benchmark Results — batch K={VMAP_K}")
print(f"{'=' * 76}")

for i, n in enumerate(grid_size):
    print(f"\n  Grid size : {n} x {n}\n")
    print(f"  {'Solver':<{W}} {'V-cycle (s)':>12}  {'V-res':>10}  {'PCG (s)':>10}  {'PCG-res':>10}")
    print(f"  {'-' * 68}")
    for name in SOLVERS:
        if len(time_vcycle[name]) > i:
            print(
                f"  {name:<{W}}"
                f"  {time_vcycle[name][i]:>12.4f}"
                f"  {res_vcycle[name][i]:>10.2e}"
                f"  {time_pcg[name][i]:>10.4f}"
                f"  {res_pcg[name][i]:>10.2e}"
            )

print(f"\n{'=' * 76}\n")



  Benchmark Results — batch K=64

  Grid size : 50 x 50

  Solver         V-cycle (s)       V-res     PCG (s)     PCG-res
  --------------------------------------------------------------------
  ruge_stuben          0.2801    4.94e-01      0.2575    7.95e-11

  Grid size : 100 x 100

  Solver         V-cycle (s)       V-res     PCG (s)     PCG-res
  --------------------------------------------------------------------
  ruge_stuben          1.2156    6.78e-01      2.1150    8.89e-11




## Plots

In [ ]:
plot_runtime(
    grid_size,
    [(name, time_vcycle[name]) for name in SOLVERS if not all(np.isnan(time_vcycle[name]))],
    VMAP_K,
    filename="solvers_vcycle_runtime.png",
    show=True,
)

plot_runtime(
    grid_size,
    [(name, time_pcg[name]) for name in SOLVERS if not all(np.isnan(time_pcg[name]))],
    VMAP_K,
    filename="solvers_pcg_runtime.png",
    show=True,
)

plot_residual(
    grid_size,
    [(name, res_vcycle[name]) for name in SOLVERS if not all(np.isnan(res_vcycle[name]))],
    filename="solvers_vcycle_residual.png",
    show=True,
)

plot_residual(
    grid_size,
    [(name, res_pcg[name]) for name in SOLVERS if not all(np.isnan(res_pcg[name]))],
    filename="solvers_pcg_residual.png",
    show=True,
)
